# 02 NLP Pipeline

This notebook cleans `customer_feedback`, inspects vocabulary differences, and generates the text features used by the deployed CLV model.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.pipeline import build_text_projection, load_dataset, preprocess_dataframe
from src.text_features import top_words_by_mask

bundle = load_dataset()
df, _ = preprocess_dataframe(bundle.data)
df[["customer_feedback", "feedback_clean", "feedback_missing"]].head()

In [ ]:
df["feedback_word_count"].describe()

In [ ]:
positive_words = top_words_by_mask(df["customer_feedback"], df["nps_score"] >= 9)
negative_words = top_words_by_mask(df["customer_feedback"], df["nps_score"] <= 4)
positive_words[:15], negative_words[:15]

In [ ]:
tfidf, svd, tfidf_features = build_text_projection(df["feedback_clean"])
print("Vocabulary size:", len(tfidf.get_feature_names_out()))
print("SVD components:", tfidf_features.shape[1])
print("Explained variance:", round(float(svd.explained_variance_ratio_.sum()), 4))
tfidf_features.head()

In [ ]:
df[[
    "sentiment_polarity",
    "sentiment_subjectivity",
    "has_negation",
    "churn_language",
    "praise_language",
    "feedback_word_count",
]].describe()